**Wojciech Olejnik**

# NCB do rozpoznawania spamu i niespamu

In [1]:
from sklearn.metrics import accuracy_score
import numpy as np

In [2]:
def load_data(filename, x_shape, y_shape):
    try:
        fp = open(filename)
        lines = fp.readlines()
        
        X_train = np.zeros((x_shape, y_shape))
        for index, row in enumerate(lines):
            doc, word, frequency = row.split(' ')
            X_train[int(doc)-1][int(word)-1] = int(frequency)

        return X_train
    finally:
        fp.close()

In [3]:
# train-features
X_train = load_data('data_prepared/train-features.txt', 700, 2500)
y_train = np.loadtxt('data_prepared/train-labels.txt', dtype=int)

# train-features-50
X_train_50 = load_data('data_prepared/train-features-50.txt', 50, 2498)
y_train_50 = np.loadtxt('data_prepared/train-labels-50.txt', dtype=int)

# train-features-100
X_train_100 = load_data('data_prepared/train-features-100.txt', 100, 2499)
y_train_100 = np.loadtxt('data_prepared/train-labels-100.txt', dtype=int)

# train-features-400
X_train_400 = load_data('data_prepared/train-features-400.txt', 400, 2500)
y_train_400 = np.loadtxt('data_prepared/train-labels-400.txt', dtype=int)

# test
X_test = load_data('data_prepared/test-features.txt', 260, 2500)
y_test = np.loadtxt('data_prepared/test-labels.txt', dtype=int)

In [4]:
class NaiveBayes:
    def __init__(self, y_train):
        self.classes= np.unique(y_train)
        self.prior = []
        self.loglikelihood = []
        self.Vocabulary = []

    def train(self, X_train, y_train):        
        all_doc_class = {} # wszystkie dokumenty danej klasy
        loglikelihood = {}
        prior = {}
        n_doc = len(y_train) # liczba dokumentów w zbiorze

        for c in self.classes:
            n_c = np.sum(y_train == c) # liczba dokumentów z dokumentów danej klasy
            prior[c] = np.log(n_c / n_doc)
            all_doc_class[c] = X_train[y_train == c, :] # dokument dla każej klasy
            for word in range(X_train.shape[1]):
                self.Vocabulary.append(word) # tworzymy słownik ze wszystkim wyrazami
                count_frequency = np.sum(all_doc_class[c][:, word]) # Jak czesto wystepuje dane słowo
                likehood = self.cal_likelihood(count_frequency, all_doc_class[c])
                loglikelihood[word, c] = np.log(likehood)
        self.prior = prior # wiarygoność apriori
        self.loglikelihood = loglikelihood

    def classification(self, X):
        result = []
        i = 0
        while i < X.shape[0]:
            
            one_test_doc = X[i, :]
            sum_loglikelihood = np.empty((len(self.classes)))

            for c in self.classes:
                sum_loglikelihood[c] = self.prior[c] # suma apriori dla danej klasy
                for word in range(X.shape[1]): # dla każdego słowa
                    if word in self.Vocabulary:
                        sum_loglikelihood[c] += self.loglikelihood[word, c] * one_test_doc[word] # dodanie prawdopodobieństwa
            i = i + 1
            result.append(np.argmax(sum_loglikelihood))
        return np.array(result)
    
    def cal_likelihood(self, freq, cls):
        return (freq + 1) / (np.sum(cls + 1)) #Dodajemy poprawke Laplace'a
        

In [5]:
nb = NaiveBayes(y_train)
nb.train(X_train, y_train)
y_pred = nb.classification(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Dokładnośc klasyfikacji: {accuracy * 100} %')
print(f'Poprawnie zaklasyfikowane próbki : {sum(y_test == y_pred)}')
print(f'Wartość błędnie zaklasyfikowanych próbek : {(1 - accuracy) *100} %')
print(f'Błędnie zaklasyfikowane próbki : {sum(y_test != y_pred)}')

Dokładnośc klasyfikacji: 96.92307692307692 %
Poprawnie zaklasyfikowane próbki : 252
Wartość błędnie zaklasyfikowanych próbek : 3.076923076923077 %
Błędnie zaklasyfikowane próbki : 8


In [6]:
nb = NaiveBayes(y_train_50)
nb.train(X_train_50, y_train_50)
y_pred = nb.classification(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Dokładnośc klasyfikacji: {accuracy * 100} %')
print(f'Poprawnie zaklasyfikowane próbki : {sum(y_test == y_pred)}')
print(f'Wartość błędnie zaklasyfikowanych próbek : {(1 - accuracy) *100} %')
print(f'Błędnie zaklasyfikowane próbki : {sum(y_test != y_pred)}')

Dokładnośc klasyfikacji: 62.69230769230769 %
Poprawnie zaklasyfikowane próbki : 163
Wartość błędnie zaklasyfikowanych próbek : 37.30769230769231 %
Błędnie zaklasyfikowane próbki : 97


In [7]:
nb = NaiveBayes(y_train_100)
nb.train(X_train_100, y_train_100)
y_pred = nb.classification(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Dokładnośc klasyfikacji: {accuracy * 100} %')
print(f'Poprawnie zaklasyfikowane próbki : {sum(y_test == y_pred)}')
print(f'Wartość błędnie zaklasyfikowanych próbek : {(1 - accuracy) *100} %')
print(f'Błędnie zaklasyfikowane próbki : {sum(y_test != y_pred)}')

Dokładnośc klasyfikacji: 94.61538461538461 %
Poprawnie zaklasyfikowane próbki : 246
Wartość błędnie zaklasyfikowanych próbek : 5.384615384615388 %
Błędnie zaklasyfikowane próbki : 14


In [8]:
nb = NaiveBayes(y_train_400)
nb.train(X_train_400, y_train_400)
y_pred = nb.classification(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Dokładnośc klasyfikacji: {accuracy * 100} %')
print(f'Poprawnie zaklasyfikowane próbki : {sum(y_test == y_pred)}')
print(f'Wartość błędnie zaklasyfikowanych próbek : {(1 - accuracy) *100} %')
print(f'Błędnie zaklasyfikowane próbki : {sum(y_test != y_pred)}')

Dokładnośc klasyfikacji: 95.76923076923077 %
Poprawnie zaklasyfikowane próbki : 249
Wartość błędnie zaklasyfikowanych próbek : 4.230769230769226 %
Błędnie zaklasyfikowane próbki : 11


# Podsumowanie

| liczność zbiór uczącego | dokładność | ilość poprawnie sklasyfikowanych | Błąd klasyfikacji | ilość błędnie sklasyfikowanych |
|-------------------------|------------|----------------------------------|-------------------|--------------------------------|
| 50                      | 62.69 %    | 163                              | 37.31 %           | 97                             |
| 100                     | 94.61 %    | 246                              | 5.39 %            | 14                             |
| 400                     | 95.76 %    | 249                              | 4.24 %            | 11                             |
| 700 (pełny)             | 96.92 %    | 252                              | 3.08 %            | 8                              |

**Wnioski:**
Ze wzorstem rozmiaru danych uczących, dokładność klasyfikatora zwiększała się, co powodowało zmieniejsze się ilości błędnie sklasyfikowanych próbek. Dla najmniej liczniejszego zbioru (50 dokumentów) błąd klasyfikacji wynosił 37.31 % ( 97 próbek błędnie sklasyfikowanych). Najlepszą dokładność uzyskano dla zbioru 700 dokumnetów, która wynosiła 96.92% i tylko 8 próbek zostało błędnie skasyfikowane.